In [0]:
from pyspark.sql import functions as F

trips = spark.table("urban_mobility.silver.trips_current")
drivers = spark.table("urban_mobility.bronze.drivers").select("driver_id", "rating", "status")
vehicles = spark.table("urban_mobility.bronze.vehicles").select("vehicle_id", "make", "model", "vehicle_type", "capacity")
pickup_zones = spark.table("urban_mobility.bronze.zones").select(
    F.col("zone_id").alias("pickup_zone_id"),
    F.col("zone_name").alias("pickup_zone_name"),
    F.col("borough").alias("pickup_borough")
)
dropoff_zones = spark.table("urban_mobility.bronze.zones").select(
    F.col("zone_id").alias("dropoff_zone_id"),
    F.col("zone_name").alias("dropoff_zone_name"),
    F.col("borough").alias("dropoff_borough")
)

enriched = (
    trips
    .join(drivers, "driver_id", "left")
    .join(vehicles, "vehicle_id", "left")
    .join(pickup_zones, "pickup_zone_id", "left")
    .join(dropoff_zones, "dropoff_zone_id", "left")
    .withColumn(
        "trip_duration_minutes",
        F.when(
            F.col("dropoff_datetime").isNotNull() & F.col("pickup_datetime").isNotNull(),
            F.round((F.col("dropoff_datetime").cast("long") - F.col("pickup_datetime").cast("long")) / 60.0, 2)
        )
    )
    .withColumn(
        "average_speed_kmh",
        F.when(
            (F.col("trip_duration_minutes").isNotNull()) & (F.col("trip_duration_minutes") > 0) & F.col("distance_km").isNotNull(),
            F.round(F.col("distance_km") / (F.col("trip_duration_minutes") / 60.0), 2)
        )
    )
    .withColumn(
        "revenue_per_km",
        F.when(
            F.col("distance_km").isNotNull() & (F.col("distance_km") > 0) & F.col("total_amount").isNotNull(),
            F.round(F.col("total_amount") / F.col("distance_km"), 2)
        )
    )
    .withColumn("pickup_hour", F.hour("pickup_datetime"))
    .withColumn("pickup_day_of_week", F.date_format("pickup_datetime", "EEEE"))
    .withColumn(
        "is_weekend",
        F.dayofweek("pickup_datetime").isin(1, 7)
    )
    .withColumn(
        "is_peak_hour",
        F.col("pickup_hour").isin(7, 8, 9, 16, 17, 18, 19)
    )
)

enriched.write.mode("overwrite").format("delta").saveAsTable("urban_mobility.silver.trips_enriched")

result = spark.table("urban_mobility.silver.trips_enriched")
print("silver.trips_enriched rows:", result.count())
result.select("trip_id", "trip_duration_minutes", "average_speed_kmh", "revenue_per_km", "pickup_hour", "is_weekend", "is_peak_hour").show(5)